# 01 - Production: MACE-MP-0

There are no methodological questions here: the parameters have already been
validated in notebook `00`. This notebook loops over the materials and writes
the JSON files to `data/results/mace/`. It is intentionally short and repetitive.

**Separate environment** (`requirements/mace.txt`): the three software stacks
cannot coexist in the same environment. `mace_mp(default_dtype="float64")`
calls `torch.set_default_dtype`, which is global state: instantiating CHGNet
after MACE in the same session causes the first linear layer to fail with
`expected mat1 and mat2 to have the same dtype`.
Restart the kernel between models.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.paths import RESULTS, REFERENCE, FIGURES
import src.materials as M
print("ROOT:", ROOT)

ROOT: c:\Users\antoj\Desktop\Tonio\projects\mlip-phonon-benchmark


In [ ]:
from mace.calculators import mace_mp
import mace
calc = mace_mp(model="medium", default_dtype="float64", device="cuda")
MODEL_NAME = "MACE-MP-0 medium (2023-12-03-mace-128-L1_epoch-199.model)"

MODEL_VERSION = (
    f"{MODEL_NAME}; "
    f"mace={mace.__version__}"
)
print("MACE", MODEL_VERSION)

c:\Users\antoj\AppData\Local\Programs\Python\Python311\Lib\site-packages\e3nn\o3\_wigner.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  _Jd, _W3j_flat, _W3j_indices = 

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Using Materials Project MACE for MACECalculator with C:\Users\antoj\.cache\mace/20231203mace128L1_epoch199model
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


c:\Users\antoj\AppData\Local\Programs\Python\Python311\Lib\site-packages\mace\calculators\mace.py:197: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(f=model_path,

Using head Default out of ['Default']
MACE 0.3.14


In [3]:
import time, json
from src.phonons import compute_phonons, thermal_properties, band_structure, make_band_path
from src.io_utils import make_payload, save_result
from src.reference import load_reference
from src.phonons import ph_to_ase

MODEL = "mace"
DTYPE = "float64"
FMAX = 0.005
DISP = 0.01

materials = list(M.MATERIALS)      # or M.QUICK for a quick test
print("materials:", materials)

for key in materials:
    out = RESULTS / MODEL / f"{key}.json"
    if out.exists():
        print(f"[skip] {key} already computed")
        continue
    try:
        t0 = time.perf_counter()
        
        mp_id = M.MATERIALS[key]["mp_id"]

        ph_ref = load_reference(
            mp_id,
            functional="pbe",
            use_nac=False,
        )

        band_path = make_band_path(
            ph_ref,
            npoints=101,
        )

        atoms = ph_to_ase(ph_ref.unitcell)
        ph, relaxed = compute_phonons(
            atoms,
            calc,
            supercell_matrix=ph_ref.supercell_matrix,
            primitive_matrix=ph_ref.primitive_matrix,
            disp=DISP,
            fmax=FMAX,
            fix_symmetry=True,
            logfile=None,
        )
        tp = thermal_properties(ph)
        bands = band_structure(ph, band_path)
        payload = make_payload(key, MODEL, MODEL_VERSION, DTYPE, ph, relaxed,
                               DISP, FMAX, tp, bands,
                               runtime_s=round(time.perf_counter() - t0, 2))
        save_result(MODEL, key, payload)
        print(f"[ok] {key:5s} omega_max={payload['omega_max_THz']:7.3f} THz  "
              f"imag={payload['has_imaginary']}  ({payload['runtime_s']}s)")
    except Exception as e:
        print(f"[FAIL] {key}: {type(e).__name__}: {e}")

materials: ['Si', 'SiC', 'AlP', 'ZnS', 'MgO', 'NaCl', 'AlN', 'GaN']
[ok] Si    omega_max= 11.190 THz  imag=False  (4.87s)
[ok] SiC   omega_max= 21.876 THz  imag=False  (6.76s)
[ok] AlP   omega_max= 10.509 THz  imag=False  (1.9s)
[ok] ZnS   omega_max=  8.620 THz  imag=False  (1.99s)
[ok] MgO   omega_max= 15.023 THz  imag=False  (2.25s)
[ok] NaCl  omega_max=  5.515 THz  imag=False  (2.23s)
[ok] AlN   omega_max= 19.986 THz  imag=False  (7.65s)
[ok] GaN   omega_max= 18.616 THz  imag=False  (7.54s)
